In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_recipes_gsheet.py
=====================
Colab batch runner for RECIPE-graph extraction (GPT-5.1) working DIRECTLY on
your trajectories Google Sheet - no more download-csv / edit / re-upload cycle.
Writes: recipe graph -> column F, recipe status -> column G, per row, live.

How it works:
  - You keep your data in a normal Google Sheet (edit it in the browser any time).
  - Colab authenticates with your Google account (one click, no keys to paste
    besides the Azure one) and reads the rows straight from the sheet.
  - After each link finishes, ONLY that row's graph+status cells are written
    back to the sheet. Your manual edits to other rows are never touched,
    and you can watch statuses appear live while the run is going.

ONE-TIME SETUP (2 minutes):
  1. In Drive, right-click trajectories.csv (or recipes.csv) -> Open with
     -> Google Sheets. Then in that Sheet: File -> Save as Google Sheets.
     This creates the native Sheet you'll use from now on.
  2. Copy the Sheet's URL from the browser bar into SHEET_URL below.
  3. Run this script. A Google popup asks you to allow access - click through.

RULE OF THUMB: don't hand-edit the graph/status columns of rows the runner is
currently processing; anything else you can edit freely, even mid-run.
"""

import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("openai", "openai"), ("requests", "requests"),
                   ("trafilatura", "trafilatura"), ("bs4", "beautifulsoup4"),
                   ("json_repair", "json-repair"), ("gspread", "gspread")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        print(f"Installing {_pkg} ..."); _pip([_pkg])

import json, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests
from openai import OpenAI
import gspread

try:
    import trafilatura; _HAVE_TRAFILATURA = True
except Exception:
    _HAVE_TRAFILATURA = False
try:
    from bs4 import BeautifulSoup; _HAVE_BS4 = True
except Exception:
    _HAVE_BS4 = False
import json_repair

# =====================================================================
# 1. CONFIG - edit these
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_recipe_extraction.txt"      # must contain {{INTERVIEW_TEXT}}

# The Google Sheet to work on (paste the browser URL of the Sheet)
SHEET_URL      = "https://docs.google.com/spreadsheets/d/1DPLl0KVl_0zuEmHSQZIDigdBp7W3rvR6I9pAp2KY-Ps/edit?gid=0#gid=0"
WORKSHEET_NAME = None            # None = first tab; or e.g. "Sheet1"

# Column layout (0-based indices, same convention as before)
LINK_COL   = 0
GRAPH_COL  = 3                  # where THIS run writes its graph
STATUS_COL = 4                  # where THIS run writes its status
HAS_HEADER = True

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
AZURE_API_KEY   = ""

MAX_WORKERS     = 5
MAX_OUTPUT_TOK  = 8000
MAX_INPUT_CHARS = 45000
HTTP_TIMEOUT    = 30
MAX_RETRIES     = 5
OVERWRITE       = False          # False = skip rows that already have a graph (silently)

SHEET_CELL_LIMIT = 49500         # Google Sheets hard limit is 50,000 chars/cell
PLACEHOLDER = "{{INTERVIEW_TEXT}}"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

# =====================================================================
# 2. GOOGLE AUTH + OPEN SHEET
# =====================================================================
def open_sheet():
    from google.colab import auth as colab_auth  # type: ignore
    colab_auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    if "PASTE_YOUR" in SHEET_URL:
        raise SystemExit("Set SHEET_URL in section 1 to your Google Sheet's URL.")
    u = SHEET_URL.strip()
    if "drive.google.com/file" in u or u.endswith(".csv") or u.endswith(".xlsx"):
        raise SystemExit(
            "That is a Drive FILE link, not a Google Sheet.\n"
            "Open the file with Google Sheets, do File -> Save as Google Sheets,\n"
            "then copy the URL that looks like docs.google.com/spreadsheets/d/...")
    try:
        sh = gc.open_by_url(u) if "spreadsheets/d/" in u else gc.open_by_key(u)
    except Exception as exc:
        raise SystemExit(f"Could not open the sheet: {str(exc)[:200]}\nURL given: {u[:100]}")
    ws = sh.worksheet(WORKSHEET_NAME) if WORKSHEET_NAME else sh.get_worksheet(0)
    print(f"[sheet] opened '{sh.title}' / tab '{ws.title}'")
    return ws

def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        pass

def build_client() -> OpenAI:
    key = (AZURE_API_KEY or "").strip()
    if not key:
        raise SystemExit("AZURE_API_KEY is empty. Paste your Azure API key in section 1.")
    return OpenAI(base_url=AZURE_ENDPOINT, api_key=key)

# =====================================================================
# 3. FETCH + CLEAN PAGE TEXT (direct -> Jina fallback)
# =====================================================================
def _fetch_direct(url):
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    html = resp.text
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    return text

def _fetch_jina(url):
    resp = requests.get("https://r.jina.ai/" + url,
                        headers={"User-Agent": USER_AGENT, "X-Return-Format": "text"},
                        timeout=HTTP_TIMEOUT * 2)
    resp.raise_for_status()
    return resp.text

def fetch_text(url):
    text, err = "", ""
    try:
        text = _fetch_direct(url)
    except Exception as exc:
        err = f"direct: {exc}"
    if len(text.strip()) < 400:
        try:
            jtext = _fetch_jina(url)
            if len(jtext.strip()) > len(text.strip()):
                text = jtext
        except Exception as exc:
            err = (err + " | " if err else "") + f"jina: {exc}"
    if not text.strip():
        return "", f"fetch_error: {err or 'empty page'}"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text).strip()
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]"
    return text, "ok"

# =====================================================================
# 4. CALL THE MODEL
# =====================================================================
def call_model(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(delay); delay = min(delay * 2, 60)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts: {last}")

# =====================================================================
# 5. PARSE MODEL OUTPUT (tolerant, both tag names)
# =====================================================================
def parse_graph(raw):
    block = raw.strip()
    m = re.search(r"<(trajectory_graph|recipe_graph)>(.*?)</\1>", block, re.DOTALL | re.IGNORECASE)
    if m:
        block = m.group(2).strip()
    else:
        m2 = re.search(r"\{.*\}", block, re.DOTALL)
        if m2:
            block = m2.group(0)
    block = re.sub(r"^```(?:json)?|```$", "", block.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(block), "parsed"
    except json.JSONDecodeError as exc:
        try:
            repaired = json_repair.loads(block)
            if isinstance(repaired, (dict, list)) and repaired:
                return repaired, "parsed (repaired)"
        except Exception:
            pass
        return None, f"json_decode_error: {exc}"

# =====================================================================
# 6. SHEET WRITE-BACK (per-row, with 429 retry)
# =====================================================================
def a1_col(idx0):
    """0-based column index -> A1 letter(s)."""
    s, n = "", idx0 + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s

def write_row(ws, sheet_row, graph_val, status_val, lock):
    """Write graph+status cells of one sheet row (1-based row number)."""
    c1, c2 = sorted([GRAPH_COL, STATUS_COL])
    # build a contiguous range covering both cells, preserving anything between
    rng = f"{a1_col(c1)}{sheet_row}:{a1_col(c2)}{sheet_row}"
    values = [""] * (c2 - c1 + 1)
    values[GRAPH_COL - c1] = graph_val
    values[STATUS_COL - c1] = status_val
    delay = 2.0
    for attempt in range(6):
        try:
            with lock:
                if c2 - c1 == 1:      # adjacent columns - single clean update
                    ws.update(rng, [values], value_input_option="RAW")
                else:                  # non-adjacent - two cell updates
                    ws.update_cell(sheet_row, GRAPH_COL + 1, graph_val)
                    ws.update_cell(sheet_row, STATUS_COL + 1, status_val)
            return True
        except Exception as exc:
            if "429" in str(exc) or "quota" in str(exc).lower():
                time.sleep(delay); delay = min(delay * 2, 30)
                continue
            print(f"    !! sheet write failed row {sheet_row}: {str(exc)[:100]}")
            return False
    return False

# =====================================================================
# 7. MAIN
# =====================================================================
def main():
    mount_drive()
    template = Path(PROMPT_FILE).read_text(encoding="utf-8") if Path(PROMPT_FILE).exists() else ""
    if not template:
        raise SystemExit(f"Prompt file not found: {PROMPT_FILE}")
    if PLACEHOLDER not in template:
        raise SystemExit(f"Prompt file must contain {PLACEHOLDER}.")

    ws = open_sheet()
    all_rows = ws.get_all_values()   # list of lists of strings
    if not all_rows:
        raise SystemExit("Sheet is empty.")
    start = 1 if HAS_HEADER else 0
    width = max(LINK_COL, GRAPH_COL, STATUS_COL) + 1
    data = [r + [""] * (width - len(r)) if len(r) < width else r for r in all_rows]

    if HAS_HEADER:
        hdr_updates = []
        if (data[0][GRAPH_COL] or "").strip() != GRAPH_HEADER:
            hdr_updates.append((1, GRAPH_COL + 1, GRAPH_HEADER))
        if (data[0][STATUS_COL] or "").strip() != STATUS_HEADER:
            hdr_updates.append((1, STATUS_COL + 1, STATUS_HEADER))
        for r, c, v in hdr_updates:
            ws.update_cell(r, c, v)

    # decide which rows to process - skipped rows are never written to
    todo, skipped = [], 0
    for i in range(start, len(data)):
        link = (data[i][LINK_COL] or "").strip()
        if not link:
            skipped += 1; continue
        if (data[i][GRAPH_COL] or "").strip() and not OVERWRITE:
            skipped += 1; continue
        todo.append(i)

    client = build_client()
    print(f"\n[run] {len(todo)} link(s) to process | {skipped} skipped untouched "
          f"| workers={MAX_WORKERS} | model={DEPLOYMENT_NAME}")
    print(f"[run] graph -> col {a1_col(GRAPH_COL)} ('{GRAPH_HEADER}') | "
          f"status -> col {a1_col(STATUS_COL)} ('{STATUS_HEADER}')\n")

    counts, counts_lock, sheet_lock, done = {}, threading.Lock(), threading.Lock(), 0

    def work(idx):
        link = (data[idx][LINK_COL] or "").strip()
        sheet_row = idx + 1                      # sheets are 1-based
        text, fnote = fetch_text(link)
        if not text:
            write_row(ws, sheet_row, "", f"fetch_failed: {fnote}"[:400], sheet_lock)
            return idx, "fetch_failed"
        try:
            raw = call_model(client, template.replace(PLACEHOLDER, text))
        except Exception as exc:
            write_row(ws, sheet_row, "", f"api_failed: {str(exc)[:150]}", sheet_lock)
            return idx, "api_failed"
        graph, pnote = parse_graph(raw)
        if graph is None:
            write_row(ws, sheet_row, raw.strip()[:SHEET_CELL_LIMIT],
                      f"parse_failed: {pnote}"[:400], sheet_lock)
            return idx, "parse_failed"
        blob = json.dumps(graph, ensure_ascii=False)
        n_nodes = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
        if len(blob) > SHEET_CELL_LIMIT:
            # graph too big for one cell -> save full JSON to Drive, keep pointer in cell
            overflow_dir = Path(DRIVE_FOLDER) / "graphs_overflow"
            overflow_dir.mkdir(exist_ok=True)
            fpath = overflow_dir / f"row{sheet_row}.json"
            fpath.write_text(blob, encoding="utf-8")
            write_row(ws, sheet_row, f"@file:{fpath.name}",
                      f"ok ({n_nodes} nodes) [stored in graphs_overflow/{fpath.name} - "
                      f"{len(blob)} chars exceeds sheet cell limit]", sheet_lock)
            return idx, "ok_overflow"
        status = f"ok ({n_nodes} nodes)" + (" [repaired]" if "repaired" in pnote else "")
        write_row(ws, sheet_row, blob, status, sheet_lock)
        return idx, "ok"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(work, i): i for i in todo}
        for fut in as_completed(futures):
            idx, status = fut.result()
            with counts_lock:
                counts[status] = counts.get(status, 0) + 1
                done += 1
            print(f"[{done}/{len(todo)}] {status:12s} {(data[idx][LINK_COL] or '')[:80]}")

    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print("[done] results are already in the Google Sheet - nothing to download/upload.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[sheet] opened 'recipes_final' / tab 'Sheet1'

[run] 1 link(s) to process | 59 skipped untouched | workers=5 | model=gpt-5.1
[run] graph -> col D ('recipe_graph') | status -> col E ('recipe_status')



/tmp/ipykernel_554/2413525223.py:247: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(rng, [values], value_input_option="RAW")


[1/1] ok           https://review.firstround.com/the-founder-dating-playbook-heres-the-process-i-us

[done] ok=1
[done] results are already in the Google Sheet - nothing to download/upload.


In [ ]:
# =====================================================================
# CELL 2 (v2) — retry failed rows, gentle on the Jina reader
# Run AFTER your main cell. Reuses build_client, call_model,
# parse_graph, read_table, pad, and CONFIG constants from cell 1.
# =====================================================================
import csv, json, re, time, threading, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# --- retry settings ---
RETRY_WORKERS = 2      # keep LOW so the Jina reader doesn't rate-limit (429)
USE_JINA      = True
JINA_API_KEY  = ""   # <=== paste your free Jina key
JINA_TRIES    = 4      # attempts against Jina, with backoff on 429/5xx


def _clean(t):
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n\s*\n\s*\n+", "\n\n", t).strip()
    return t


def _extract_html(html):
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    return _clean(text)


def _fetch_via_jina(url):
    headers = {"User-Agent": USER_AGENT, "Accept": "text/plain", "X-Return-Format": "text"}
    if JINA_API_KEY.strip() and "PUT_YOUR_KEY" not in JINA_API_KEY:
        headers["Authorization"] = f"Bearer {JINA_API_KEY.strip()}"
    delay, last = 3.0, "jina_failed"
    for _ in range(JINA_TRIES):
        try:
            r = requests.get("https://r.jina.ai/" + url, headers=headers, timeout=HTTP_TIMEOUT + 60)
            if r.status_code == 200 and r.text:
                return _clean(r.text), "ok"
            last = f"jina_http_{r.status_code}"
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(delay); delay = min(delay * 2, 40); continue
            return "", last            # hard error (401/403/451...) — no point retrying
        except Exception as exc:
            last = f"jina_error: {exc}"
            time.sleep(delay); delay = min(delay * 2, 40)
    return "", last


def fetch_text(url):   # overrides cell 1
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }
    text, err = "", ""
    for _ in range(2):
        try:
            resp = requests.get(url, headers=headers, timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text:
                text = _extract_html(resp.text)
                if len(text) >= 400:
                    return text[:MAX_INPUT_CHARS] if MAX_INPUT_CHARS else text, "ok"
                err = f"thin_direct ({len(text)} chars)"
            else:
                err = f"http_{resp.status_code}"
        except Exception as exc:
            err = f"fetch_error: {exc}"
        time.sleep(1.5)

    if USE_JINA:
        jtext, jnote = _fetch_via_jina(url)
        if len(jtext) >= 400:
            return (jtext[:MAX_INPUT_CHARS] if MAX_INPUT_CHARS else jtext), "ok(jina)"
        if jtext and len(jtext) > len(text):
            text = jtext
        err = jnote if not text else err

    if not text:
        return "", (err or "empty")
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS]
    return text, f"thin_text ({len(text)} chars)"


# --- reload + select failed rows ---
template = Path(PROMPT_FILE).read_text(encoding="utf-8")
header, data = read_table(LINKS_FILE)
width = max(LINK_COL, GRAPH_COL, STATUS_COL if STATUS_COL is not None else 0) + 1
for row in data:
    pad(row, width)
if header is not None:
    pad(header, width)

out_csv = Path(OUTPUT_CSV) if OUTPUT_CSV else Path(LINKS_FILE).with_name(
    Path(LINKS_FILE).stem + "_with_graphs.csv")
save_lock = threading.Lock()


def save():
    with save_lock:
        with out_csv.open("w", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh)
            if header is not None:
                w.writerow(header)
            w.writerows(data)


def _valid(cell):
    cell = (cell or "").strip()
    if not cell:
        return False
    try:
        obj = json.loads(cell)
        return isinstance(obj, dict) and bool(obj.get("nodes"))
    except Exception:
        return False


todo = [i for i, row in enumerate(data)
        if (row[LINK_COL] or "").strip() and not _valid(row[GRAPH_COL])]

client = build_client()
print(f"[retry] {len(todo)} row(s) | workers={RETRY_WORKERS} | "
      f"jina_key={'yes' if JINA_API_KEY.strip() and 'PUT_YOUR_KEY' not in JINA_API_KEY else 'no'}\n")

counts, clock, done = {}, threading.Lock(), 0


def work(idx):
    row = data[idx]
    link = (row[LINK_COL] or "").strip()
    text, fnote = fetch_text(link)
    if not text:
        row[GRAPH_COL] = ""
        if STATUS_COL is not None:
            row[STATUS_COL] = f"fetch_failed: {fnote}"
        return idx, f"fetch_failed ({fnote})"
    try:
        raw = call_model(client, template.replace(PLACEHOLDER, text))
    except Exception:
        row[GRAPH_COL] = ""
        if STATUS_COL is not None:
            row[STATUS_COL] = "api_failed"
        return idx, "api_failed"
    graph, pnote = parse_graph(raw)
    if graph is None:
        row[GRAPH_COL] = raw.strip()
        if STATUS_COL is not None:
            row[STATUS_COL] = f"parse_failed: {pnote}"
        return idx, "parse_failed"
    row[GRAPH_COL] = json.dumps(graph, ensure_ascii=False)
    n = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
    if STATUS_COL is not None:
        row[STATUS_COL] = f"ok ({n} nodes)" + (" [repaired]" if "repaired" in pnote else "")
    return idx, f"ok ({n} nodes)"


with ThreadPoolExecutor(max_workers=RETRY_WORKERS) as pool:
    futures = {pool.submit(work, i): i for i in todo}
    for fut in as_completed(futures):
        idx, status = fut.result()
        with clock:
            done += 1
            key = status.split()[0]
            counts[key] = counts.get(key, 0) + 1
        print(f"[{done}/{len(todo)}] {status:28s} {(data[idx][LINK_COL] or '')[:70]}")
        save()

save()
print("\n[retry done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
print(f"[retry done] -> {out_csv}")

[auth] Using Azure API key authentication.
[retry] 6 row(s) | workers=2 | jina_key=yes



ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None


[1/6] ok (0 nodes)                 https://www.cs.cmu.edu/~harchol/gradschooltalk.pdf
[2/6] ok (0 nodes)                 https://www.cs.jhu.edu/~aadelucia/2025/03/31/apply-phd/
[3/6] ok (52 nodes)                https://writingcenter.unc.edu/tips-and-tools/application-essays/
[4/6] ok (36 nodes)                https://gradschool.cornell.edu/admissions/application-steps/statements
[5/6] ok (21 nodes)                https://gradschool.cornell.edu/inclusion/recruitment/prospective-stude
[6/6] ok (44 nodes)                https://www.findaphd.com/guides/phd-interview-questions

[retry done] ok=6
[retry done] -> /content/drive/MyDrive/QCRI_Project/Robee_Dataset/recipes.csv


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_recipes_colab.py
====================
Colab batch runner (PARALLEL) for RECIPE-graph extraction with Azure GPT-5.1,
using API-KEY authentication (no browser / device-code login).

CSV layout (by column position):
    column 1  -> link          (input)
    column 2  -> title         (input, kept as-is)
    column 3  -> domain        (input, kept as-is)
    column 5  -> recipe_graph  (OUTPUT — the recipe graph JSON is written here)
    column 6  -> graph_status  (OUTPUT — ok / fetch_failed / api_failed / parse_failed)

For every link it: fetches the page text, drops it into the {{INTERVIEW_TEXT}} slot of
cot_recipe_prompt.txt, calls GPT-5.1, and writes the returned recipe graph JSON into
column 4 of the same row. Rows run CONCURRENTLY; the CSV is re-saved after each row so
you can stop and resume.

------------------------------------------------------------------------------
AUTHENTICATION — Azure API key.
Paste your key in AZURE_API_KEY below. No Microsoft login / device code is used.
SECURITY: never share the key in chat/screenshots. If it leaks, regenerate it in the
Azure portal (Keys and Endpoint -> Regenerate).
------------------------------------------------------------------------------
"""

# =====================================================================
# 0. DEPENDENCIES
# =====================================================================
import importlib
import subprocess
import sys


def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


for _mod, _pkg in [
    ("openai", "openai"),
    ("requests", "requests"),
    ("trafilatura", "trafilatura"),
    ("bs4", "beautifulsoup4"),
    ("json_repair", "json-repair"),
]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        print(f"Installing {_pkg} ...")
        _pip([_pkg])

import csv
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from openai import OpenAI

try:
    import trafilatura
    _HAVE_TRAFILATURA = True
except Exception:
    _HAVE_TRAFILATURA = False
try:
    from bs4 import BeautifulSoup
    _HAVE_BS4 = True
except Exception:
    _HAVE_BS4 = False
import json_repair


# =====================================================================
# 1. CONFIG  —  edit these
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_recipe_extraction.txt"   # must contain {{INTERVIEW_TEXT}}
LINKS_FILE   = f"{DRIVE_FOLDER}/trajectories.csv"             # link | title | domain | (graph)
OUTPUT_CSV   = LINKS_FILE                                # writes back into the same file in place

# Column layout (0-based indices)
LINK_COL   = 0      # 1st column = the link to process
GRAPH_COL  = 6     # 4th column = where the recipe graph JSON is written
STATUS_COL = 57      # 5th column = status note (set to None to disable)
HAS_HEADER = True # True if the first CSV row is a header row

# Header labels used only when that header cell is empty.
GRAPH_HEADER  = "recipe_graph"
STATUS_HEADER = "graph_status"

# =====================================================================
# Azure GPT-5.1 — API KEY AUTHENTICATION
# =====================================================================
AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
# IMPORTANT: paste your (regenerated) Azure API key here. Do NOT commit/share it.
AZURE_API_KEY   = ""

# Run settings
MAX_WORKERS     = 5       # parallel requests (raise cautiously; watch for rate limits)
MAX_OUTPUT_TOK  = 8000    # max tokens the model may return
MAX_INPUT_CHARS = 45000   # cap page text sent to the model (0 = no cap)
HTTP_TIMEOUT    = 30      # seconds per page fetch
MAX_RETRIES     = 5       # API retry attempts on transient errors
OVERWRITE       = True  # re-run rows that already have a graph in column 4

PLACEHOLDER = "{{INTERVIEW_TEXT}}"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")


# =====================================================================
# 2. MOUNT DRIVE
# =====================================================================
def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        print("[drive] mounted")
    except Exception as exc:
        print(f"[drive] not mounting (not on Colab or already mounted): {exc}")


# =====================================================================
# 3. AZURE CLIENT  (API KEY AUTHENTICATION — no browser login)
# =====================================================================
def build_client() -> OpenAI:
    if not AZURE_API_KEY or AZURE_API_KEY == "YOUR_AZURE_API_KEY_HERE":
        raise SystemExit("ERROR: paste your Azure API key into AZURE_API_KEY.")
    print("[auth] Using Azure API key authentication.")
    return OpenAI(
        base_url=AZURE_ENDPOINT,
        api_key=AZURE_API_KEY,
        default_headers={"api-key": AZURE_API_KEY},
    )


# =====================================================================
# 4. READ CSV  (positional; keeps every column + row order)
# =====================================================================
def read_table(path_str):
    """Return (header_or_None, data_rows). Rows are lists of cells."""
    path = Path(path_str)
    if not path.exists():
        raise SystemExit(f"Links file not found: {path}")
    delim = "\t" if path.suffix.lower() == ".tsv" else ","
    with path.open(newline="", encoding="utf-8-sig") as fh:
        all_rows = [list(r) for r in csv.reader(fh, delimiter=delim)]
    if not all_rows:
        raise SystemExit(f"{path} is empty.")
    if HAS_HEADER:
        return all_rows[0], all_rows[1:]
    return None, all_rows


def pad(row, width):
    if len(row) < width:
        row.extend([""] * (width - len(row)))
    return row


# =====================================================================
# 5. FETCH + CLEAN PAGE TEXT
# =====================================================================
def fetch_text(url):
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
        resp.raise_for_status()
    except Exception as exc:
        return "", f"fetch_error: {exc}"
    html = resp.text
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text).strip()
    note = "ok"
    if len(text) < 400:
        note = f"thin_text ({len(text)} chars)"
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]"
    return text, note


# =====================================================================
# 6. CALL GPT-5.1  (retries; gpt-5.x uses max_completion_tokens, default temp)
# =====================================================================
def call_model(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK,
            )
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(delay)
            delay = min(delay * 2, 60)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts: {last}")


# =====================================================================
# 7. PARSE MODEL OUTPUT -> graph JSON (tolerant)
# =====================================================================
def parse_graph(raw):
    block = raw.strip()
    # accept <recipe_graph> or <trajectory_graph> tags if present; else raw JSON
    m = re.search(r"<(?:recipe|trajectory)_graph>(.*?)</(?:recipe|trajectory)_graph>",
                  block, re.DOTALL | re.IGNORECASE)
    if m:
        block = m.group(1).strip()
    else:
        m2 = re.search(r"\{.*\}", block, re.DOTALL)
        if m2:
            block = m2.group(0)
    block = re.sub(r"^```(?:json)?|```$", "", block.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(block), "parsed"
    except json.JSONDecodeError as exc:
        try:
            repaired = json_repair.loads(block)
            if isinstance(repaired, (dict, list)) and repaired:
                return repaired, "parsed (repaired)"
        except Exception:
            pass
        return None, f"json_decode_error: {exc}"


# =====================================================================
# 8. MAIN  (parallel; writes the recipe graph into column 4)
# =====================================================================
def main():
    mount_drive()

    template = Path(PROMPT_FILE).read_text(encoding="utf-8") if Path(PROMPT_FILE).exists() else ""
    if not template:
        raise SystemExit(f"Prompt file not found: {PROMPT_FILE}")
    if PLACEHOLDER not in template:
        raise SystemExit(f"Prompt file must contain {PLACEHOLDER}.")

    header, data = read_table(LINKS_FILE)
    if not data:
        raise SystemExit("No data rows to process.")

    width = max(LINK_COL, GRAPH_COL, STATUS_COL if STATUS_COL is not None else 0) + 1
    for row in data:
        pad(row, width)
    if header is not None:
        pad(header, width)
        header[GRAPH_COL] = header[GRAPH_COL] or GRAPH_HEADER
        if STATUS_COL is not None:
            header[STATUS_COL] = header[STATUS_COL] or STATUS_HEADER

    src = Path(LINKS_FILE)
    out_csv = Path(OUTPUT_CSV) if OUTPUT_CSV else src.with_name(f"{src.stem}_with_graphs.csv")

    save_lock = threading.Lock()

    def save():
        with save_lock:
            with out_csv.open("w", newline="", encoding="utf-8") as fh:
                w = csv.writer(fh)
                if header is not None:
                    w.writerow(header)
                w.writerows(data)

    def set_status(row, val):
        if STATUS_COL is not None:
            row[STATUS_COL] = val

    # decide which rows to process
    todo = []
    for idx, row in enumerate(data):
        link = (row[LINK_COL] or "").strip()
        if not link:
            set_status(row, "no_link")
            continue
        if (row[GRAPH_COL] or "").strip() and not OVERWRITE:
            set_status(row, (row[STATUS_COL] if STATUS_COL is not None else "") or "already_done")
            continue
        todo.append(idx)

    client = build_client()
    print(f"\n[run] {len(todo)} link(s) | workers={MAX_WORKERS} | model={DEPLOYMENT_NAME}")
    print(f"[run] recipe graph -> column {GRAPH_COL + 1} | output -> {out_csv}\n")

    counts, counts_lock, done = {}, threading.Lock(), 0

    def work(idx):
        row = data[idx]
        link = (row[LINK_COL] or "").strip()
        text, fnote = fetch_text(link)
        if not text:
            row[GRAPH_COL] = ""
            set_status(row, f"fetch_failed: {fnote}")
            return idx, "fetch_failed"
        try:
            raw = call_model(client, template.replace(PLACEHOLDER, text))
        except Exception:
            row[GRAPH_COL] = ""
            set_status(row, "api_failed")
            return idx, "api_failed"
        graph, pnote = parse_graph(raw)
        if graph is None:
            row[GRAPH_COL] = raw.strip()          # keep raw so nothing is lost
            set_status(row, f"parse_failed: {pnote}")
            return idx, "parse_failed"
        row[GRAPH_COL] = json.dumps(graph, ensure_ascii=False)   # compact one-cell JSON
        n_nodes = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
        set_status(row, f"ok ({n_nodes} nodes)" + (" [repaired]" if "repaired" in pnote else ""))
        return idx, "ok"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(work, i): i for i in todo}
        for fut in as_completed(futures):
            idx, status = fut.result()
            with counts_lock:
                counts[status] = counts.get(status, 0) + 1
                done += 1
            print(f"[{done}/{len(todo)}] {status:12s} {(data[idx][LINK_COL] or '')[:80]}")
            save()

    save()
    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print(f"[done] CSV with recipe graphs -> {out_csv}")


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[drive] mounted
[auth] Using Azure API key authentication.

[run] 60 link(s) | workers=5 | model=gpt-5.1
[run] recipe graph -> column 7 | output -> /content/drive/MyDrive/QCRI_Project/Robee_Dataset/trajectories.csv

[1/60] ok           https://bennokrojer.github.io/far_away.html
[2/60] ok           https://towardsdatascience.com/my-journey-to-phd-admission-in-artificial-intelli
[3/60] ok           https://blog.clementfung.me/ubc-recap
[4/60] ok           https://thephdplace.com/prep-a-non-traditional-route-to-phd/
[5/60] ok           https://stevenkolawole.medium.com/the-road-to-phd-ii-my-approach-to-the-applicat
[6/60] ok           https://thinktown.medium.com/alumna-story-from-cmu-engineering-student-to-mit-ph
[7/60] ok           https://studyinternational.com/news/stanford-university-holy-spirit
[8/60] ok           https://sites.manchester.ac.uk/humanities

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_recipes_gsheet_tab2.py
==========================
Recipe-graph extraction (GPT-5.1) that READS links from the first tab of your
Google Sheet and WRITES results to the SECOND tab as three columns:
    A: link | B: recipe_graph | C: node_count
Output rows are aligned to the same row number as the input link, so writes are
live and per-row (no append races). Set MAX_ROWS to iterate on the first N links.
"""

import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("openai", "openai"), ("requests", "requests"),
                   ("trafilatura", "trafilatura"), ("bs4", "beautifulsoup4"),
                   ("json_repair", "json-repair"), ("gspread", "gspread")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        print(f"Installing {_pkg} ..."); _pip([_pkg])

import json, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests
from openai import OpenAI
import gspread

try:
    import trafilatura; _HAVE_TRAFILATURA = True
except Exception:
    _HAVE_TRAFILATURA = False
try:
    from bs4 import BeautifulSoup; _HAVE_BS4 = True
except Exception:
    _HAVE_BS4 = False
import json_repair

# =====================================================================
# 1. CONFIG - edit these
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_recipe_extraction_6.txt"      # must contain {{INTERVIEW_TEXT}}

SHEET_URL       = "https://docs.google.com/spreadsheets/d/1DPLl0KVl_0zuEmHSQZIDigdBp7W3rvR6I9pAp2KY-Ps/edit?gid=0#gid=0"
INPUT_TAB       = 0              # first tab: where the links live (0-based index)
OUTPUT_TAB_NAME = "recipe_output"  # second tab: created if missing
LINK_COL        = 0              # column A on the INPUT tab holds the link
HAS_HEADER      = True
MAX_ROWS        = 5              # process only the first N links (None = all)

# OUTPUT tab column layout (0-based)
OUT_LINK_COL  = 0                # A: link
OUT_GRAPH_COL = 1                # B: recipe_graph
OUT_COUNT_COL = 2                # C: node_count
OUT_HEADERS   = ["link", "recipe_graph", "node_count"]

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
AZURE_API_KEY   = ""

MAX_WORKERS     = 5
MAX_OUTPUT_TOK  = 12000          # bumped up: long pages (Lucy/Karpathy) + full graph
MAX_INPUT_CHARS = 45000
HTTP_TIMEOUT    = 30
MAX_RETRIES     = 5
OVERWRITE       = True           # True while iterating so reruns replace old graphs

SHEET_CELL_LIMIT = 49500
PLACEHOLDER = "{{INTERVIEW_TEXT}}"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

# =====================================================================
# 2. GOOGLE AUTH + OPEN SHEET (input tab + output tab)
# =====================================================================
def open_sheets():
    from google.colab import auth as colab_auth  # type: ignore
    colab_auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    u = SHEET_URL.strip()
    if "PASTE_YOUR" in u:
        raise SystemExit("Set SHEET_URL in section 1 to your Google Sheet's URL.")
    if "drive.google.com/file" in u or u.endswith(".csv") or u.endswith(".xlsx"):
        raise SystemExit("That is a Drive FILE link, not a Google Sheet. Save as Google Sheets first.")
    try:
        sh = gc.open_by_url(u) if "spreadsheets/d/" in u else gc.open_by_key(u)
    except Exception as exc:
        raise SystemExit(f"Could not open the sheet: {str(exc)[:200]}\nURL: {u[:100]}")
    ws_in = sh.get_worksheet(INPUT_TAB)
    # get-or-create the output tab
    try:
        ws_out = sh.worksheet(OUTPUT_TAB_NAME)
    except gspread.WorksheetNotFound:
        ws_out = sh.add_worksheet(title=OUTPUT_TAB_NAME, rows=max(100, ws_in.row_count), cols=10)
        print(f"[sheet] created output tab '{OUTPUT_TAB_NAME}'")
    print(f"[sheet] '{sh.title}': input tab '{ws_in.title}' -> output tab '{ws_out.title}'")
    return ws_in, ws_out

def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        pass

def build_client() -> OpenAI:
    key = (AZURE_API_KEY or "").strip()
    if not key:
        raise SystemExit("AZURE_API_KEY is empty. Paste your Azure API key in section 1.")
    return OpenAI(base_url=AZURE_ENDPOINT, api_key=key)

# =====================================================================
# 3. FETCH + CLEAN PAGE TEXT (direct -> Jina fallback)
# =====================================================================
def _fetch_direct(url):
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    html = resp.text
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    return text

def _fetch_jina(url):
    resp = requests.get("https://r.jina.ai/" + url,
                        headers={"User-Agent": USER_AGENT, "X-Return-Format": "text"},
                        timeout=HTTP_TIMEOUT * 2)
    resp.raise_for_status()
    return resp.text

def fetch_text(url):
    text, err = "", ""
    try:
        text = _fetch_direct(url)
    except Exception as exc:
        err = f"direct: {exc}"
    if len(text.strip()) < 400:
        try:
            jtext = _fetch_jina(url)
            if len(jtext.strip()) > len(text.strip()):
                text = jtext
        except Exception as exc:
            err = (err + " | " if err else "") + f"jina: {exc}"
    if not text.strip():
        return "", f"fetch_error: {err or 'empty page'}"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text).strip()
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]"
    return text, "ok"

# =====================================================================
# 4. CALL THE MODEL
# =====================================================================
def call_model(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(delay); delay = min(delay * 2, 60)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts: {last}")

# =====================================================================
# 5. PARSE MODEL OUTPUT (tolerant, both tag names)
# =====================================================================
def parse_graph(raw):
    block = raw.strip()
    m = re.search(r"<(trajectory_graph|recipe_graph)>(.*?)</\1>", block, re.DOTALL | re.IGNORECASE)
    if m:
        block = m.group(2).strip()
    else:
        m2 = re.search(r"\{.*\}", block, re.DOTALL)
        if m2:
            block = m2.group(0)
    block = re.sub(r"^```(?:json)?|```$", "", block.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(block), "parsed"
    except json.JSONDecodeError as exc:
        try:
            repaired = json_repair.loads(block)
            if isinstance(repaired, (dict, list)) and repaired:
                return repaired, "parsed (repaired)"
        except Exception:
            pass
        return None, f"json_decode_error: {exc}"

# =====================================================================
# 6. OUTPUT-TAB WRITE-BACK (link + graph + node_count, one row)
# =====================================================================
def a1_col(idx0):
    s, n = "", idx0 + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s

def write_out_row(ws, sheet_row, link, graph_val, count_val, lock):
    """Write link+graph+count into cols A:C of the output tab at sheet_row (1-based)."""
    cols = [OUT_LINK_COL, OUT_GRAPH_COL, OUT_COUNT_COL]
    c1, c2 = min(cols), max(cols)
    values = [""] * (c2 - c1 + 1)
    values[OUT_LINK_COL  - c1] = link
    values[OUT_GRAPH_COL - c1] = graph_val
    values[OUT_COUNT_COL - c1] = count_val
    rng = f"{a1_col(c1)}{sheet_row}:{a1_col(c2)}{sheet_row}"
    delay = 2.0
    for _ in range(6):
        try:
            with lock:
                ws.update(rng, [values], value_input_option="RAW")
            return True
        except Exception as exc:
            if "429" in str(exc) or "quota" in str(exc).lower():
                time.sleep(delay); delay = min(delay * 2, 30); continue
            print(f"    !! output write failed row {sheet_row}: {str(exc)[:100]}")
            return False
    return False

# =====================================================================
# 7. MAIN
# =====================================================================
def main():
    mount_drive()
    template = Path(PROMPT_FILE).read_text(encoding="utf-8") if Path(PROMPT_FILE).exists() else ""
    if not template:
        raise SystemExit(f"Prompt file not found: {PROMPT_FILE}")
    if PLACEHOLDER not in template:
        raise SystemExit(f"Prompt file must contain {PLACEHOLDER}.")

    ws_in, ws_out = open_sheets()

    in_rows = ws_in.get_all_values()
    if not in_rows:
        raise SystemExit("Input tab is empty.")
    width = LINK_COL + 1
    in_rows = [r + [""] * (width - len(r)) if len(r) < width else r for r in in_rows]

    # header on the output tab
    ws_out.update(f"A1:{a1_col(OUT_COUNT_COL)}1", [OUT_HEADERS], value_input_option="RAW")

    # existing output graphs (to honor OVERWRITE=False)
    out_rows = ws_out.get_all_values()
    out_rows = [r + [""] * (OUT_GRAPH_COL + 1 - len(r)) if len(r) < OUT_GRAPH_COL + 1 else r
                for r in out_rows]

    start = 1 if HAS_HEADER else 0
    todo, skipped = [], 0
    for i in range(start, len(in_rows)):
        link = (in_rows[i][LINK_COL] or "").strip()
        if not link:
            skipped += 1; continue
        existing = out_rows[i][OUT_GRAPH_COL] if i < len(out_rows) else ""
        if (existing or "").strip() and not OVERWRITE:
            skipped += 1; continue
        todo.append(i)
        if MAX_ROWS and len(todo) >= MAX_ROWS:
            break

    client = build_client()
    print(f"\n[run] {len(todo)} link(s) to process | {skipped} skipped "
          f"| workers={MAX_WORKERS} | model={DEPLOYMENT_NAME}")
    print(f"[run] output tab '{ws_out.title}': "
          f"{a1_col(OUT_LINK_COL)}=link  {a1_col(OUT_GRAPH_COL)}=recipe_graph  "
          f"{a1_col(OUT_COUNT_COL)}=node_count\n")

    counts, counts_lock, sheet_lock, done = {}, threading.Lock(), threading.Lock(), 0

    def work(idx):
        link = (in_rows[idx][LINK_COL] or "").strip()
        sheet_row = idx + 1
        text, fnote = fetch_text(link)
        if not text:
            write_out_row(ws_out, sheet_row, link, "", f"fetch_failed: {fnote}"[:200], sheet_lock)
            return idx, "fetch_failed"
        try:
            raw = call_model(client, template.replace(PLACEHOLDER, text))
        except Exception as exc:
            write_out_row(ws_out, sheet_row, link, "", f"api_failed: {str(exc)[:120]}", sheet_lock)
            return idx, "api_failed"
        graph, pnote = parse_graph(raw)
        if graph is None:
            write_out_row(ws_out, sheet_row, link, raw.strip()[:SHEET_CELL_LIMIT],
                          f"parse_failed", sheet_lock)
            return idx, "parse_failed"
        n_nodes = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
        blob = json.dumps(graph, ensure_ascii=False)
        if len(blob) > SHEET_CELL_LIMIT:
            overflow_dir = Path(DRIVE_FOLDER) / "graphs_overflow"
            overflow_dir.mkdir(exist_ok=True)
            fpath = overflow_dir / f"row{sheet_row}.json"
            fpath.write_text(blob, encoding="utf-8")
            write_out_row(ws_out, sheet_row, link, f"@file:{fpath.name}", n_nodes, sheet_lock)
            return idx, "ok_overflow"
        write_out_row(ws_out, sheet_row, link, blob, n_nodes, sheet_lock)
        return idx, "ok"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(work, i): i for i in todo}
        for fut in as_completed(futures):
            idx, status = fut.result()
            with counts_lock:
                counts[status] = counts.get(status, 0) + 1
                done += 1
            print(f"[{done}/{len(todo)}] {status:12s} {(in_rows[idx][LINK_COL] or '')[:80]}")

    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print(f"[done] results are in the '{ws_out.title}' tab (link | recipe_graph | node_count).")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[sheet] 'recipes_final': input tab 'Sheet1' -> output tab 'recipe_output'


/tmp/ipykernel_3397/2285457485.py:265: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws_out.update(f"A1:{a1_col(OUT_COUNT_COL)}1", [OUT_HEADERS], value_input_option="RAW")



[run] 5 link(s) to process | 0 skipped | workers=5 | model=gpt-5.1
[run] output tab 'recipe_output': A=link  B=recipe_graph  C=node_count



/tmp/ipykernel_3397/2285457485.py:236: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(rng, [values], value_input_option="RAW")


[1/5] ok           https://gradschool.duke.edu/professional-development/blog/your-guide-strong-phd-
[2/5] ok           https://www.rootedinstem.com/grad-school/t6eex9ljigqf53uodg8sl9pd02i22r
[3/5] ok           https://sophiepomme.com/how-to-apply-for-a-phd-everything-you-need-to-succeed
[4/5] ok           https://karpathy.github.io/2016/09/07/phd/
[5/5] ok           https://lucylai.com/blog/gradapps.html

[done] ok=5
[done] results are in the 'recipe_output' tab (link | recipe_graph | node_count).


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_causal_gsheet_tab3.py
=========================
Causal-graph extraction (GPT-5.1). For each row it takes the RECIPE GRAPH from the
'recipe_output' tab (col B) plus the original page TEXT (re-fetched from the link in col A),
fills the prompt's recipe slot ({{RECIPE_GRAPH}}) and its article-text slot (auto-detected),
and writes a new 'causal_output' tab:
    A: link | B: causal_graph | C: node_count | D: edge_count
Output rows are aligned to the recipe_output row numbers. Set MAX_ROWS to iterate on first N.
"""

import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("openai", "openai"), ("requests", "requests"),
                   ("trafilatura", "trafilatura"), ("bs4", "beautifulsoup4"),
                   ("json_repair", "json-repair"), ("gspread", "gspread")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        print(f"Installing {_pkg} ..."); _pip([_pkg])

import json, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests
from openai import OpenAI
import gspread

try:
    import trafilatura; _HAVE_TRAFILATURA = True
except Exception:
    _HAVE_TRAFILATURA = False
try:
    from bs4 import BeautifulSoup; _HAVE_BS4 = True
except Exception:
    _HAVE_BS4 = False
import json_repair

# =====================================================================
# 1. CONFIG - edit these
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_causal_extraction_6.txt"   # must contain {{RECIPE_GRAPH}} + one article-text slot

SHEET_URL        = "https://docs.google.com/spreadsheets/d/1DPLl0KVl_0zuEmHSQZIDigdBp7W3rvR6I9pAp2KY-Ps/edit?gid=0#gid=0"
RECIPE_TAB_NAME  = "recipe_output"    # read recipe graphs from here
CAUSAL_TAB_NAME  = "causal_output"    # write causal graphs here (created if missing)
HAS_HEADER       = True
MAX_ROWS         = 5                  # process only first N recipe rows (None = all)

# recipe_output (input) columns, 0-based  -> matches the recipe runner's output
IN_LINK_COL   = 0                     # A: link
IN_GRAPH_COL  = 1                     # B: recipe_graph  (JSON, or "@file:rowN.json" pointer)

# causal_output (output) columns, 0-based
OUT_LINK_COL  = 0                     # A: link
OUT_GRAPH_COL = 1                     # B: causal_graph
OUT_NODES_COL = 2                     # C: node_count
OUT_EDGES_COL = 3                     # D: edge_count
OUT_HEADERS   = ["link", "causal_graph", "node_count", "edge_count"]

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
AZURE_API_KEY   = ""

MAX_WORKERS     = 5
MAX_OUTPUT_TOK  = 12000
MAX_INPUT_CHARS = 45000
HTTP_TIMEOUT    = 30
MAX_RETRIES     = 5
OVERWRITE       = True                # True while iterating so reruns replace old graphs

SHEET_CELL_LIMIT = 49500
PH_RECIPE = "{{RECIPE_GRAPH}}"        # recipe-graph slot (fixed)
PH_TEXT   = "{{SOURCE_TEXT}}"         # article-text slot; auto-overridden to match your prompt
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

# =====================================================================
# 2. GOOGLE AUTH + OPEN SHEETS
# =====================================================================
def open_sheets():
    from google.colab import auth as colab_auth  # type: ignore
    colab_auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    u = SHEET_URL.strip()
    if "drive.google.com/file" in u or u.endswith(".csv") or u.endswith(".xlsx"):
        raise SystemExit("That is a Drive FILE link, not a Google Sheet. Save as Google Sheets first.")
    sh = gc.open_by_url(u) if "spreadsheets/d/" in u else gc.open_by_key(u)
    try:
        ws_in = sh.worksheet(RECIPE_TAB_NAME)
    except gspread.WorksheetNotFound:
        raise SystemExit(f"Input tab '{RECIPE_TAB_NAME}' not found. Run the recipe extractor first.")
    try:
        ws_out = sh.worksheet(CAUSAL_TAB_NAME)
    except gspread.WorksheetNotFound:
        ws_out = sh.add_worksheet(title=CAUSAL_TAB_NAME, rows=max(100, ws_in.row_count), cols=10)
        print(f"[sheet] created output tab '{CAUSAL_TAB_NAME}'")
    print(f"[sheet] '{sh.title}': '{ws_in.title}' -> '{ws_out.title}'")
    return ws_in, ws_out

def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        pass

def build_client() -> OpenAI:
    key = (AZURE_API_KEY or "").strip()
    if not key:
        raise SystemExit("AZURE_API_KEY is empty. Paste your Azure API key in section 1.")
    return OpenAI(base_url=AZURE_ENDPOINT, api_key=key)

# =====================================================================
# 3. FETCH + CLEAN PAGE TEXT (direct -> Jina fallback)
# =====================================================================
def _fetch_direct(url):
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    html = resp.text
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    return text

def _fetch_jina(url):
    resp = requests.get("https://r.jina.ai/" + url,
                        headers={"User-Agent": USER_AGENT, "X-Return-Format": "text"},
                        timeout=HTTP_TIMEOUT * 2)
    resp.raise_for_status()
    return resp.text

def fetch_text(url):
    text, err = "", ""
    try:
        text = _fetch_direct(url)
    except Exception as exc:
        err = f"direct: {exc}"
    if len(text.strip()) < 400:
        try:
            jtext = _fetch_jina(url)
            if len(jtext.strip()) > len(text.strip()):
                text = jtext
        except Exception as exc:
            err = (err + " | " if err else "") + f"jina: {exc}"
    if not text.strip():
        return "", f"fetch_error: {err or 'empty page'}"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text).strip()
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]"
    return text, "ok"

# =====================================================================
# 4. LOAD RECIPE GRAPH FROM THE INPUT CELL (handles @file overflow)
# =====================================================================
def load_recipe(cell_value):
    v = (cell_value or "").strip()
    if not v:
        return "", "empty_recipe_cell"
    if v.startswith("@file:"):
        fpath = Path(DRIVE_FOLDER) / "graphs_overflow" / v[len("@file:"):]
        if not fpath.exists():
            return "", f"overflow_file_missing: {fpath.name}"
        return fpath.read_text(encoding="utf-8"), "ok"
    return v, "ok"   # already JSON text

# =====================================================================
# 5. CALL THE MODEL
# =====================================================================
def call_model(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(delay); delay = min(delay * 2, 60)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts: {last}")

# =====================================================================
# 6. PARSE MODEL OUTPUT (tolerant)
# =====================================================================
def parse_graph(raw):
    block = raw.strip()
    m = re.search(r"<(causal_graph|recipe_graph|trajectory_graph)>(.*?)</\1>",
                  block, re.DOTALL | re.IGNORECASE)
    if m:
        block = m.group(2).strip()
    else:
        m2 = re.search(r"\{.*\}", block, re.DOTALL)
        if m2:
            block = m2.group(0)
    block = re.sub(r"^```(?:json)?|```$", "", block.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(block), "parsed"
    except json.JSONDecodeError as exc:
        try:
            repaired = json_repair.loads(block)
            if isinstance(repaired, (dict, list)) and repaired:
                return repaired, "parsed (repaired)"
        except Exception:
            pass
        return None, f"json_decode_error: {exc}"

# =====================================================================
# 7. OUTPUT-TAB WRITE-BACK
# =====================================================================
def a1_col(idx0):
    s, n = "", idx0 + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s

def write_out_row(ws, sheet_row, link, graph_val, nodes_val, edges_val, lock):
    cols = [OUT_LINK_COL, OUT_GRAPH_COL, OUT_NODES_COL, OUT_EDGES_COL]
    c1, c2 = min(cols), max(cols)
    values = [""] * (c2 - c1 + 1)
    values[OUT_LINK_COL  - c1] = link
    values[OUT_GRAPH_COL - c1] = graph_val
    values[OUT_NODES_COL - c1] = nodes_val
    values[OUT_EDGES_COL - c1] = edges_val
    rng = f"{a1_col(c1)}{sheet_row}:{a1_col(c2)}{sheet_row}"
    delay = 2.0
    for _ in range(6):
        try:
            with lock:
                ws.update(rng, [values], value_input_option="RAW")
            return True
        except Exception as exc:
            if "429" in str(exc) or "quota" in str(exc).lower():
                time.sleep(delay); delay = min(delay * 2, 30); continue
            print(f"    !! output write failed row {sheet_row}: {str(exc)[:100]}")
            return False
    return False

# =====================================================================
# 8. MAIN
# =====================================================================
def main():
    mount_drive()
    template = Path(PROMPT_FILE).read_text(encoding="utf-8") if Path(PROMPT_FILE).exists() else ""
    if not template:
        raise SystemExit(f"Prompt file not found: {PROMPT_FILE}")

    # --- auto-detect placeholders: recipe-graph slot + article-text slot ---
    global PH_TEXT
    if PH_RECIPE not in template:
        raise SystemExit(f"Prompt file must contain {PH_RECIPE}.")
    other = [p for p in dict.fromkeys(re.findall(r"\{\{[^}]+\}\}", template)) if p != PH_RECIPE]
    if not other:
        raise SystemExit("Prompt has no placeholder for the fetched article text "
                         "(found only the recipe slot). Add e.g. {{SOURCE_TEXT}} where the article should go.")
    PH_TEXT = other[0]
    if len(other) > 1:
        print(f"[prompt] WARNING: multiple non-recipe placeholders {other}; using {PH_TEXT} for the article text.")
    print(f"[prompt] recipe graph -> {PH_RECIPE} | fetched article text -> {PH_TEXT}")

    ws_in, ws_out = open_sheets()
    in_rows = ws_in.get_all_values()
    if not in_rows:
        raise SystemExit(f"Input tab '{RECIPE_TAB_NAME}' is empty. Run the recipe extractor first.")
    width = max(IN_LINK_COL, IN_GRAPH_COL) + 1
    in_rows = [r + [""] * (width - len(r)) if len(r) < width else r for r in in_rows]

    ws_out.update(f"A1:{a1_col(OUT_EDGES_COL)}1", [OUT_HEADERS], value_input_option="RAW")
    out_rows = ws_out.get_all_values()
    out_rows = [r + [""] * (OUT_GRAPH_COL + 1 - len(r)) if len(r) < OUT_GRAPH_COL + 1 else r
                for r in out_rows]

    start = 1 if HAS_HEADER else 0
    todo, skipped = [], 0
    for i in range(start, len(in_rows)):
        link  = (in_rows[i][IN_LINK_COL] or "").strip()
        graph = (in_rows[i][IN_GRAPH_COL] or "").strip()
        if not link or not graph:          # need both a link and a recipe graph
            skipped += 1; continue
        existing = out_rows[i][OUT_GRAPH_COL] if i < len(out_rows) else ""
        if (existing or "").strip() and not OVERWRITE:
            skipped += 1; continue
        todo.append(i)
        if MAX_ROWS and len(todo) >= MAX_ROWS:
            break

    client = build_client()
    print(f"\n[run] {len(todo)} recipe row(s) to lift into causal graphs | {skipped} skipped "
          f"| workers={MAX_WORKERS} | model={DEPLOYMENT_NAME}")
    print(f"[run] output '{ws_out.title}': A=link B=causal_graph C=node_count D=edge_count\n")

    counts, counts_lock, sheet_lock, done = {}, threading.Lock(), threading.Lock(), 0

    def work(idx):
        link = (in_rows[idx][IN_LINK_COL] or "").strip()
        sheet_row = idx + 1
        recipe_json, rnote = load_recipe(in_rows[idx][IN_GRAPH_COL])
        if not recipe_json:
            write_out_row(ws_out, sheet_row, link, "", f"recipe_load_failed: {rnote}", "", sheet_lock)
            return idx, "recipe_load_failed"
        text, fnote = fetch_text(link)
        if not text:
            write_out_row(ws_out, sheet_row, link, "", f"fetch_failed: {fnote}"[:200], "", sheet_lock)
            return idx, "fetch_failed"
        prompt = template.replace(PH_RECIPE, recipe_json).replace(PH_TEXT, text)
        try:
            raw = call_model(client, prompt)
        except Exception as exc:
            write_out_row(ws_out, sheet_row, link, "", f"api_failed: {str(exc)[:120]}", "", sheet_lock)
            return idx, "api_failed"
        graph, pnote = parse_graph(raw)
        if graph is None:
            write_out_row(ws_out, sheet_row, link, raw.strip()[:SHEET_CELL_LIMIT],
                          "parse_failed", "", sheet_lock)
            return idx, "parse_failed"
        n_nodes = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
        n_edges = len(graph.get("edges", [])) if isinstance(graph, dict) else 0
        blob = json.dumps(graph, ensure_ascii=False)
        if len(blob) > SHEET_CELL_LIMIT:
            overflow_dir = Path(DRIVE_FOLDER) / "causal_overflow"
            overflow_dir.mkdir(exist_ok=True)
            fpath = overflow_dir / f"row{sheet_row}.json"
            fpath.write_text(blob, encoding="utf-8")
            write_out_row(ws_out, sheet_row, link, f"@file:{fpath.name}", n_nodes, n_edges, sheet_lock)
            return idx, "ok_overflow"
        write_out_row(ws_out, sheet_row, link, blob, n_nodes, n_edges, sheet_lock)
        return idx, "ok"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(work, i): i for i in todo}
        for fut in as_completed(futures):
            idx, status = fut.result()
            with counts_lock:
                counts[status] = counts.get(status, 0) + 1
                done += 1
            print(f"[{done}/{len(todo)}] {status:16s} {(in_rows[idx][IN_LINK_COL] or '')[:70]}")

    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print(f"[done] causal graphs are in the '{ws_out.title}' tab.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[prompt] recipe graph -> {{RECIPE_GRAPH}} | fetched article text -> {{SOURCE_TEXT}}
[sheet] 'recipes_final': 'recipe_output' -> 'causal_output'


/tmp/ipykernel_2802/1782065714.py:292: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws_out.update(f"A1:{a1_col(OUT_EDGES_COL)}1", [OUT_HEADERS], value_input_option="RAW")



[run] 5 recipe row(s) to lift into causal graphs | 0 skipped | workers=5 | model=gpt-5.1
[run] output 'causal_output': A=link B=causal_graph C=node_count D=edge_count



/tmp/ipykernel_2802/1782065714.py:254: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(rng, [values], value_input_option="RAW")


[1/5] ok               https://gradschool.duke.edu/professional-development/blog/your-guide-s
[2/5] ok               https://sophiepomme.com/how-to-apply-for-a-phd-everything-you-need-to-
[3/5] ok               https://karpathy.github.io/2016/09/07/phd/
[4/5] ok               https://www.rootedinstem.com/grad-school/t6eex9ljigqf53uodg8sl9pd02i22
[5/5] ok               https://lucylai.com/blog/gradapps.html

[done] ok=5
[done] causal graphs are in the 'causal_output' tab.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
run_causal_gsheet_tab3.py
=========================
Causal-graph extraction (GPT-5.1). For each row it takes the RECIPE GRAPH from the
'recipe_output' tab (col B) plus the original page TEXT (re-fetched from the link in col A),
fills the prompt's recipe slot ({{RECIPE_GRAPH}}) and its article-text slot (auto-detected),
and writes the 'causal_output' tab:
    A: link | B: causal_graph | C: node_count | D: edge_count
Output rows are aligned to the recipe_output row numbers. Set MAX_ROWS to iterate on first N.

v7 extraction: each causal super-vertex now carries the recipe nodes collapsed inside it
(`covers` with id/type/verbatim label, `internal_edges`, entry/exit ids) so a click in the
visualizer can expand it. Coverage is checked per row and printed in the run log.
"""

import importlib, subprocess, sys

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for _mod, _pkg in [("openai", "openai"), ("requests", "requests"),
                   ("trafilatura", "trafilatura"), ("bs4", "beautifulsoup4"),
                   ("json_repair", "json-repair"), ("gspread", "gspread")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        print(f"Installing {_pkg} ..."); _pip([_pkg])

import json, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests
from openai import OpenAI
import gspread

try:
    import trafilatura; _HAVE_TRAFILATURA = True
except Exception:
    _HAVE_TRAFILATURA = False
try:
    from bs4 import BeautifulSoup; _HAVE_BS4 = True
except Exception:
    _HAVE_BS4 = False
import json_repair

# =====================================================================
# 1. CONFIG - edit these
# =====================================================================
DRIVE_FOLDER = "/content/drive/MyDrive/QCRI_Project/Robee_Dataset"
PROMPT_FILE  = f"{DRIVE_FOLDER}/prompt_causal_extraction_7.txt"   # must contain {{RECIPE_GRAPH}} + one article-text slot

SHEET_URL        = "https://docs.google.com/spreadsheets/d/1DPLl0KVl_0zuEmHSQZIDigdBp7W3rvR6I9pAp2KY-Ps/edit?gid=0#gid=0"
RECIPE_TAB_NAME  = "recipe_output"    # read recipe graphs from here
CAUSAL_TAB_NAME  = "causal_output"    # write causal graphs here (created if missing)
HAS_HEADER       = True
MAX_ROWS         = 7                 # process only first N recipe rows (None = all)

# recipe_output (input) columns, 0-based  -> matches the recipe runner's output
IN_LINK_COL   = 0                     # A: link
IN_GRAPH_COL  = 1                     # B: recipe_graph  (JSON, or "@file:rowN.json" pointer)

# causal_output (output) columns, 0-based
OUT_LINK_COL  = 0                     # A: link
OUT_GRAPH_COL = 1                     # B: causal_graph
OUT_NODES_COL = 2                     # C: node_count
OUT_EDGES_COL = 3                     # D: edge_count
OUT_HEADERS   = ["link", "causal_graph", "node_count", "edge_count"]

AZURE_ENDPOINT  = "https://qcri-sakina.services.ai.azure.com/openai/v1"
DEPLOYMENT_NAME = "gpt-5.1"
AZURE_API_KEY   = ""


MAX_WORKERS     = 5
MAX_OUTPUT_TOK  = 20000               # v7 graphs carry every collapsed recipe node + label
MAX_INPUT_CHARS = 45000
HTTP_TIMEOUT    = 30
MAX_RETRIES     = 5
OVERWRITE       = True                # True while iterating so reruns replace old graphs

SHEET_CELL_LIMIT = 49500
PH_RECIPE = "{{RECIPE_GRAPH}}"        # recipe-graph slot (fixed)
PH_TEXT   = "{{SOURCE_TEXT}}"         # article-text slot; auto-overridden to match your prompt
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

# =====================================================================
# 2. GOOGLE AUTH + OPEN SHEETS
# =====================================================================
def open_sheets():
    from google.colab import auth as colab_auth  # type: ignore
    colab_auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    u = SHEET_URL.strip()
    if "drive.google.com/file" in u or u.endswith(".csv") or u.endswith(".xlsx"):
        raise SystemExit("That is a Drive FILE link, not a Google Sheet. Save as Google Sheets first.")
    sh = gc.open_by_url(u) if "spreadsheets/d/" in u else gc.open_by_key(u)
    try:
        ws_in = sh.worksheet(RECIPE_TAB_NAME)
    except gspread.WorksheetNotFound:
        raise SystemExit(f"Input tab '{RECIPE_TAB_NAME}' not found. Run the recipe extractor first.")
    try:
        ws_out = sh.worksheet(CAUSAL_TAB_NAME)
    except gspread.WorksheetNotFound:
        ws_out = sh.add_worksheet(title=CAUSAL_TAB_NAME, rows=max(100, ws_in.row_count), cols=10)
        print(f"[sheet] created output tab '{CAUSAL_TAB_NAME}'")
    print(f"[sheet] '{sh.title}': '{ws_in.title}' -> '{ws_out.title}'")
    return ws_in, ws_out

def mount_drive():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception:
        pass

def build_client() -> OpenAI:
    key = (AZURE_API_KEY or "").strip()
    if not key:
        raise SystemExit("AZURE_API_KEY is empty. Paste your Azure API key in section 1.")
    return OpenAI(base_url=AZURE_ENDPOINT, api_key=key)

# =====================================================================
# 3. FETCH + CLEAN PAGE TEXT (direct -> Jina fallback)
# =====================================================================
def _fetch_direct(url):
    resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
    resp.raise_for_status()
    html = resp.text
    text = ""
    if _HAVE_TRAFILATURA:
        try:
            text = trafilatura.extract(html, include_comments=False,
                                       include_tables=True, favor_recall=True) or ""
        except Exception:
            text = ""
    if not text and _HAVE_BS4:
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer", "aside", "noscript"]):
            t.decompose()
        text = soup.get_text("\n")
    if not text:
        text = re.sub(r"<[^>]+>", " ", html)
    return text

def _fetch_jina(url):
    resp = requests.get("https://r.jina.ai/" + url,
                        headers={"User-Agent": USER_AGENT, "X-Return-Format": "text"},
                        timeout=HTTP_TIMEOUT * 2)
    resp.raise_for_status()
    return resp.text

def fetch_text(url):
    text, err = "", ""
    try:
        text = _fetch_direct(url)
    except Exception as exc:
        err = f"direct: {exc}"
    if len(text.strip()) < 400:
        try:
            jtext = _fetch_jina(url)
            if len(jtext.strip()) > len(text.strip()):
                text = jtext
        except Exception as exc:
            err = (err + " | " if err else "") + f"jina: {exc}"
    if not text.strip():
        return "", f"fetch_error: {err or 'empty page'}"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text).strip()
    if MAX_INPUT_CHARS and len(text) > MAX_INPUT_CHARS:
        text = text[:MAX_INPUT_CHARS] + "\n\n[...TRUNCATED...]"
    return text, "ok"

# =====================================================================
# 4. LOAD RECIPE GRAPH FROM THE INPUT CELL (handles @file overflow)
# =====================================================================
def load_recipe(cell_value):
    v = (cell_value or "").strip()
    if not v:
        return "", "empty_recipe_cell"
    if v.startswith("@file:"):
        fpath = Path(DRIVE_FOLDER) / "graphs_overflow" / v[len("@file:"):]
        if not fpath.exists():
            return "", f"overflow_file_missing: {fpath.name}"
        return fpath.read_text(encoding="utf-8"), "ok"
    return v, "ok"   # already JSON text

# =====================================================================
# 5. CALL THE MODEL
# =====================================================================
def call_model(client, prompt):
    delay, last = 4.0, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=MAX_OUTPUT_TOK)
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last = exc
            if attempt == MAX_RETRIES:
                break
            time.sleep(delay); delay = min(delay * 2, 60)
    raise RuntimeError(f"API failed after {MAX_RETRIES} attempts: {last}")

# =====================================================================
# 6. PARSE MODEL OUTPUT (tolerant)
# =====================================================================
def parse_graph(raw):
    block = raw.strip()
    m = re.search(r"<(causal_graph|recipe_graph|trajectory_graph)>(.*?)</\1>",
                  block, re.DOTALL | re.IGNORECASE)
    if m:
        block = m.group(2).strip()
    else:
        m2 = re.search(r"\{.*\}", block, re.DOTALL)
        if m2:
            block = m2.group(0)
    block = re.sub(r"^```(?:json)?|```$", "", block.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(block), "parsed"
    except json.JSONDecodeError as exc:
        try:
            repaired = json_repair.loads(block)
            if isinstance(repaired, (dict, list)) and repaired:
                return repaired, "parsed (repaired)"
        except Exception:
            pass
        return None, f"json_decode_error: {exc}"

# =====================================================================
# 6b. COVERAGE CHECK — did every recipe node land in exactly one super-vertex?
#     Console-only; the sheet columns stay as they were.
# =====================================================================
def coverage_note(causal, recipe_json):
    try:
        recipe = json.loads(recipe_json)
    except Exception:
        try:
            recipe = json_repair.loads(recipe_json)
        except Exception:
            return "coverage:? (recipe unreadable)"
    if not isinstance(recipe, dict) or not isinstance(causal, dict):
        return "coverage:?"
    rids = [n.get("id") for n in (recipe.get("nodes") or []) if isinstance(n, dict)]
    rset = set(rids)

    seen, dupes, ghosts, missing_labels = [], [], [], 0
    for c in (causal.get("nodes") or []):
        for item in (c.get("covers") or []):
            rid = item.get("id") if isinstance(item, dict) else item
            if isinstance(item, dict) and not (item.get("label") or "").strip():
                missing_labels += 1
            if rid not in rset:
                ghosts.append(rid)
            elif rid in seen:
                dupes.append(rid)
            else:
                seen.append(rid)
    missing = sorted(rset - set(seen))
    n_super = len(causal.get("nodes") or [])
    ratio = round(len(rset) / max(1, n_super), 1)

    parts = [f"cover {len(seen)}/{len(rset)}", f"ratio {ratio}x"]
    if missing:
        parts.append(f"MISSING {','.join(missing[:6])}{'...' if len(missing) > 6 else ''}")
    if dupes:
        parts.append(f"DUP {len(set(dupes))}")
    if ghosts:
        parts.append(f"GHOST {len(set(ghosts))}")
    if missing_labels:
        parts.append(f"NOLABEL {missing_labels}")
    return " | ".join(parts)

# =====================================================================
# 7. OUTPUT-TAB WRITE-BACK
# =====================================================================
def a1_col(idx0):
    s, n = "", idx0 + 1
    while n:
        n, r = divmod(n - 1, 26)
        s = chr(65 + r) + s
    return s

def write_out_row(ws, sheet_row, link, graph_val, nodes_val, edges_val, lock):
    cols = [OUT_LINK_COL, OUT_GRAPH_COL, OUT_NODES_COL, OUT_EDGES_COL]
    c1, c2 = min(cols), max(cols)
    values = [""] * (c2 - c1 + 1)
    values[OUT_LINK_COL  - c1] = link
    values[OUT_GRAPH_COL - c1] = graph_val
    values[OUT_NODES_COL - c1] = nodes_val
    values[OUT_EDGES_COL - c1] = edges_val
    rng = f"{a1_col(c1)}{sheet_row}:{a1_col(c2)}{sheet_row}"
    delay = 2.0
    for _ in range(6):
        try:
            with lock:
                ws.update(rng, [values], value_input_option="RAW")
            return True
        except Exception as exc:
            if "429" in str(exc) or "quota" in str(exc).lower():
                time.sleep(delay); delay = min(delay * 2, 30); continue
            print(f"    !! output write failed row {sheet_row}: {str(exc)[:100]}")
            return False
    return False

# =====================================================================
# 8. MAIN
# =====================================================================
def main():
    mount_drive()
    template = Path(PROMPT_FILE).read_text(encoding="utf-8") if Path(PROMPT_FILE).exists() else ""
    if not template:
        raise SystemExit(f"Prompt file not found: {PROMPT_FILE}")

    # --- auto-detect placeholders: recipe-graph slot + article-text slot ---
    global PH_TEXT
    if PH_RECIPE not in template:
        raise SystemExit(f"Prompt file must contain {PH_RECIPE}.")
    other = [p for p in dict.fromkeys(re.findall(r"\{\{[^}]+\}\}", template)) if p != PH_RECIPE]
    if not other:
        raise SystemExit("Prompt has no placeholder for the fetched article text "
                         "(found only the recipe slot). Add e.g. {{SOURCE_TEXT}} where the article should go.")
    PH_TEXT = other[0]
    if len(other) > 1:
        print(f"[prompt] WARNING: multiple non-recipe placeholders {other}; using {PH_TEXT} for the article text.")
    print(f"[prompt] recipe graph -> {PH_RECIPE} | fetched article text -> {PH_TEXT}")

    ws_in, ws_out = open_sheets()
    in_rows = ws_in.get_all_values()
    if not in_rows:
        raise SystemExit(f"Input tab '{RECIPE_TAB_NAME}' is empty. Run the recipe extractor first.")
    width = max(IN_LINK_COL, IN_GRAPH_COL) + 1
    in_rows = [r + [""] * (width - len(r)) if len(r) < width else r for r in in_rows]

    ws_out.update(f"A1:{a1_col(OUT_EDGES_COL)}1", [OUT_HEADERS], value_input_option="RAW")
    out_rows = ws_out.get_all_values()
    out_rows = [r + [""] * (OUT_GRAPH_COL + 1 - len(r)) if len(r) < OUT_GRAPH_COL + 1 else r
                for r in out_rows]

    start = 1 if HAS_HEADER else 0
    todo, skipped = [], 0
    for i in range(start, len(in_rows)):
        link  = (in_rows[i][IN_LINK_COL] or "").strip()
        graph = (in_rows[i][IN_GRAPH_COL] or "").strip()
        if not link or not graph:          # need both a link and a recipe graph
            skipped += 1; continue
        existing = out_rows[i][OUT_GRAPH_COL] if i < len(out_rows) else ""
        if (existing or "").strip() and not OVERWRITE:
            skipped += 1; continue
        todo.append(i)
        if MAX_ROWS and len(todo) >= MAX_ROWS:
            break

    client = build_client()
    print(f"\n[run] {len(todo)} recipe row(s) to lift into causal graphs | {skipped} skipped "
          f"| workers={MAX_WORKERS} | model={DEPLOYMENT_NAME}")
    print(f"[run] output '{ws_out.title}': A=link B=causal_graph C=node_count D=edge_count\n")

    counts, counts_lock, sheet_lock, done = {}, threading.Lock(), threading.Lock(), 0
    notes = {}

    def work(idx):
        link = (in_rows[idx][IN_LINK_COL] or "").strip()
        sheet_row = idx + 1
        recipe_json, rnote = load_recipe(in_rows[idx][IN_GRAPH_COL])
        if not recipe_json:
            write_out_row(ws_out, sheet_row, link, "", f"recipe_load_failed: {rnote}", "", sheet_lock)
            return idx, "recipe_load_failed"
        text, fnote = fetch_text(link)
        if not text:
            write_out_row(ws_out, sheet_row, link, "", f"fetch_failed: {fnote}"[:200], "", sheet_lock)
            return idx, "fetch_failed"
        prompt = template.replace(PH_RECIPE, recipe_json).replace(PH_TEXT, text)
        try:
            raw = call_model(client, prompt)
        except Exception as exc:
            write_out_row(ws_out, sheet_row, link, "", f"api_failed: {str(exc)[:120]}", "", sheet_lock)
            return idx, "api_failed"
        graph, pnote = parse_graph(raw)
        if graph is None:
            write_out_row(ws_out, sheet_row, link, raw.strip()[:SHEET_CELL_LIMIT],
                          "parse_failed", "", sheet_lock)
            return idx, "parse_failed"

        note = coverage_note(graph, recipe_json)
        with counts_lock:
            notes[idx] = note

        n_nodes = len(graph.get("nodes", [])) if isinstance(graph, dict) else 0
        n_edges = len(graph.get("edges", [])) if isinstance(graph, dict) else 0
        blob = json.dumps(graph, ensure_ascii=False)
        if len(blob) > SHEET_CELL_LIMIT:
            overflow_dir = Path(DRIVE_FOLDER) / "causal_overflow"
            overflow_dir.mkdir(exist_ok=True)
            fpath = overflow_dir / f"row{sheet_row}.json"
            fpath.write_text(blob, encoding="utf-8")
            write_out_row(ws_out, sheet_row, link, f"@file:{fpath.name}", n_nodes, n_edges, sheet_lock)
            return idx, "ok_overflow"
        write_out_row(ws_out, sheet_row, link, blob, n_nodes, n_edges, sheet_lock)
        return idx, "ok" if "MISSING" not in note and "NOLABEL" not in note else "ok_check_coverage"

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(work, i): i for i in todo}
        for fut in as_completed(futures):
            idx, status = fut.result()
            with counts_lock:
                counts[status] = counts.get(status, 0) + 1
                done += 1
            print(f"[{done}/{len(todo)}] {status:18s} {notes.get(idx,''):38s} "
                  f"{(in_rows[idx][IN_LINK_COL] or '')[:55]}")

    print("\n[done] " + " | ".join(f"{k}={v}" for k, v in sorted(counts.items())))
    print(f"[done] causal graphs are in the '{ws_out.title}' tab.")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[prompt] recipe graph -> {{RECIPE_GRAPH}} | fetched article text -> {{SOURCE_TEXT}}
[sheet] 'recipes_final': 'recipe_output' -> 'causal_output'


/tmp/ipykernel_2802/4055315547.py:341: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws_out.update(f"A1:{a1_col(OUT_EDGES_COL)}1", [OUT_HEADERS], value_input_option="RAW")



[run] 5 recipe row(s) to lift into causal graphs | 0 skipped | workers=5 | model=gpt-5.1
[run] output 'causal_output': A=link B=causal_graph C=node_count D=edge_count



/tmp/ipykernel_2802/4055315547.py:303: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(rng, [values], value_input_option="RAW")


[1/5] ok                 cover 31/31 | ratio 4.4x               https://gradschool.duke.edu/professional-development/bl
[2/5] ok                 cover 61/61 | ratio 10.2x              https://lucylai.com/blog/gradapps.html
[3/5] ok                 cover 29/29 | ratio 3.6x               https://www.rootedinstem.com/grad-school/t6eex9ljigqf53
[4/5] ok                 cover 55/55 | ratio 6.9x               https://karpathy.github.io/2016/09/07/phd/
[5/5] ok                 cover 34/34 | ratio 4.2x               https://sophiepomme.com/how-to-apply-for-a-phd-everythi

[done] ok=5
[done] causal graphs are in the 'causal_output' tab.
